<a href="https://colab.research.google.com/github/LikhithR916/BIS_Lab/blob/main/Cuckoo%20Search%20Algorithm/1BM24CS147_CloudComputingResourceAllocation_CuckooSearchAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# =========================================================================
# 1. PROBLEM SETUP (CONFIGURABLE DATA)
# =========================================================================
# System Setup
NUM_TASKS = 8
NUM_VMS = 3

# Task Properties: [Length (Million Instructions), Memory Required (GB)]
TASKS = {
    0: {"length": 500,  "mem": 2},
    1: {"length": 1200, "mem": 4},
    2: {"length": 300,  "mem": 1},
    3: {"length": 800,  "mem": 8},
    4: {"length": 1500, "mem": 4},
    5: {"length": 600,  "mem": 2},
    6: {"length": 2000, "mem": 16},
    7: {"length": 400,  "mem": 2}
}

# VM Properties: [Processing Speed (MIPS), Available Memory (GB), Cost per second ($)]
VMS = {
    0: {"mips": 1000, "mem": 4,  "cost": 0.05},  # Small VM
    1: {"mips": 2000, "mem": 8,  "cost": 0.12},  # Medium VM
    2: {"mips": 4000, "mem": 16, "cost": 0.25}   # Large VM
}

# =========================================================================
# 2. FITNESS / GRADING FUNCTION
# =========================================================================
def calculate_score(allocation):
    """
    Lower score = Better solution.
    Calculates execution time (makespan) + cost, and penalizes bad layouts.
    """
    vm_times = np.zeros(NUM_VMS)
    vm_costs = np.zeros(NUM_VMS)
    penalty = 0

    for task_id, vm_id in enumerate(allocation):
        # Check if the VM has enough memory for the task
        if TASKS[task_id]["mem"] > VMS[vm_id]["mem"]:
            penalty += 5000 # Heavy penalty for breaking system constraints
            continue

        # Calculate time taken for this task on the assigned VM
        execution_time = TASKS[task_id]["length"] / VMS[vm_id]["mips"]
        vm_times[vm_id] += execution_time
        vm_costs[vm_id] += execution_time * VMS[vm_id]["cost"]

    # Overall system metric updates
    max_time = np.max(vm_times)
    total_cost = np.sum(vm_costs)

    # Combined target balance score (60% importance to speed, 40% to cost)
    final_score = (0.6 * max_time) + (0.4 * total_cost) + penalty
    return final_score

# =========================================================================
# 3. CUCKOO SEARCH IMPLEMENTATION
# =========================================================================
def cuckoo_search(n_nests=15, max_rounds=50, pa=0.25):
    # Initialize a population of random answers
    # Format: Every row is a solution; array positions represent tasks, entries contain VM IDs
    nests = np.random.randint(0, NUM_VMS, size=(n_nests, NUM_TASKS))
    scores = np.array([calculate_score(nest) for nest in nests])

    # Record the ultimate baseline champion
    best_idx = np.argmin(scores)
    best_nest = nests[best_idx].copy()
    best_score = scores[best_idx]

    # Optimization Loop
    for _ in range(max_rounds):

        # --- STEP A: Explore via Random Step Jump (Levy Flight replacement) ---
        for i in range(n_nests):
            # Create a mutated solution based off the best solution path
            random_step = np.random.normal(0, 1, size=NUM_TASKS)
            new_nest = nests[i] + random_step * (nests[i] - best_nest)

            # Keep indices whole integers within valid boundaries [0, NUM_VMS-1]
            new_nest = np.clip(np.round(new_nest), 0, NUM_VMS - 1).astype(int)
            new_score = calculate_score(new_nest)

            # Pit it against a completely random nest candidate from the pool
            j = np.random.randint(0, n_nests)
            if new_score < scores[j]:
                nests[j] = new_nest
                scores[j] = new_score

        # --- STEP B: Throw away worse allocations (Discovery Phase) ---
        for i in range(n_nests):
            if np.random.rand() < pa:
                # Mix components from two distinct random positions to try building a better dynamic alternative
                r1, r2 = np.random.randint(0, n_nests, size=2)
                alternative_nest = nests[i] + np.random.rand() * (nests[r1] - nests[r2])
                alternative_nest = np.clip(np.round(alternative_nest), 0, NUM_VMS - 1).astype(int)

                nests[i] = alternative_nest
                scores[i] = calculate_score(alternative_nest)

        # --- STEP C: Refresh Leaderboard standings ---
        current_best_idx = np.argmin(scores)
        if scores[current_best_idx] < best_score:
            best_score = scores[current_best_idx]
            best_nest = nests[current_best_idx].copy()

    return best_nest, best_score

# =========================================================================
# 4. EXECUTION
# =========================================================================
best_plan, optimal_score = cuckoo_search()

print("==================================================")
print("     OPTIMAL CLOUD TASKS TO VM ALLOCATION       ")
print("==================================================")
print(f"Calculated Solution Array Matrix Map: {best_plan}\n")

for task_id, vm_id in enumerate(best_plan):
    print(f"-> Task {task_id} (Requires: {TASKS[task_id]['mem']}GB)")
    print(f"   Mapped to VM {vm_id} (MIPS Capacity: {VMS[vm_id]['mips']}, Max Available Mem: {VMS[vm_id]['mem']}GB)\n")

print(f"Minimized Final Optimization Score (Lower is better): {optimal_score:.4f}")


     OPTIMAL CLOUD TASKS TO VM ALLOCATION       
Calculated Solution Array Matrix Map: [1 0 1 1 2 2 2 2]

-> Task 0 (Requires: 2GB)
   Mapped to VM 1 (MIPS Capacity: 2000, Max Available Mem: 8GB)

-> Task 1 (Requires: 4GB)
   Mapped to VM 0 (MIPS Capacity: 1000, Max Available Mem: 4GB)

-> Task 2 (Requires: 1GB)
   Mapped to VM 1 (MIPS Capacity: 2000, Max Available Mem: 8GB)

-> Task 3 (Requires: 8GB)
   Mapped to VM 1 (MIPS Capacity: 2000, Max Available Mem: 8GB)

-> Task 4 (Requires: 4GB)
   Mapped to VM 2 (MIPS Capacity: 4000, Max Available Mem: 16GB)

-> Task 5 (Requires: 2GB)
   Mapped to VM 2 (MIPS Capacity: 4000, Max Available Mem: 16GB)

-> Task 6 (Requires: 16GB)
   Mapped to VM 2 (MIPS Capacity: 4000, Max Available Mem: 16GB)

-> Task 7 (Requires: 2GB)
   Mapped to VM 2 (MIPS Capacity: 4000, Max Available Mem: 16GB)

Minimized Final Optimization Score (Lower is better): 0.8949
